# OULAD engagement cohorts with Aura Graph Analytics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jose-alvarado-guzman/oulad/blob/main/notebooks/aga_student_cohorts.ipynb)

Opens an **Aura Graph Analytics session** against the OULAD graph and asks one question:
*do students who engage with the same course materials share the same outcomes?*

The pipeline is

1. project the `(:Student)-[:REVIEWED_MATERIAL]->(:EducationalMaterial)` bipartite graph,
2. **node similarity** — Jaccard over shared materials — to link comparable learners,
3. **Louvain** over those similarity links to fall out engagement cohorts,
4. cross-tabulate the cohorts against `finalResult` to see whether they track outcomes,
5. **degree centrality** for the materials pulling the most attention,
6. write the cohort back onto each `(:Student)` in AuraDB.

Load the graph first with [`oulad_data_load.ipynb`](oulad_data_load.ipynb) if you have not
already.

## Before you start

Add these as Colab secrets (key icon, left sidebar) with *Notebook access* switched on:

| Secret | Required | Where it comes from |
| --- | --- | --- |
| `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD` | yes | your AuraDB instance |
| `AURA_CLIENT_ID`, `AURA_CLIENT_SECRET`, `AURA_PROJECT_ID` | yes | Aura console → your project → **API credentials** |
| `NEO4J_DATABASE` | no | defaults to the driver's database |
| `AURA_INSTANCEID` | no | derived from `NEO4J_URI` when absent |

The `AURA_*` API credentials are what open the analytics session, and they are separate from
the database login. `credentials.AGA_SECRETS` is the group that names them.

> **A session is billed compute.** It is a separate resource from your AuraDB instance and
> keeps costing until it is deleted. Step 5 gives it a 2-hour TTL as a backstop, and step 13
> deletes it — run that cell even if something above fails.

## 1. Setup

Re-running this resets the checkout to `origin/main`, discarding local changes, on the same
reasoning as the loader notebook: a throwaway clone that silently runs stale code is worse
than one that is always current.

The clone is here for one reason — this notebook imports `oulad.credentials`, so the secret
names, the resolution order and the instance-id derivation live in one place instead of
being restated here. It reads nothing else from the repository.

`requirements-aga.txt` installs only what these notebooks use, which is a much smaller set
than the ETL needs, so no session restart is required.

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/jose-alvarado-guzman/oulad.git'
REPO_DIR = '/content/oulad'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def run(*command):
    result = subprocess.run(command, text=True, capture_output=True)
    print((result.stdout + result.stderr).strip())
    result.check_returncode()

if IN_COLAB:
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        run('git', '-C', REPO_DIR, 'fetch', '--depth', '1', 'origin', 'main')
        run('git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main')
        run('git', '-C', REPO_DIR, 'clean', '-fd')
    else:
        run('git', 'clone', '--depth', '1', REPO_URL, REPO_DIR)
    run('git', '-C', REPO_DIR, 'log', '-1', '--format=%h %ad %s', '--date=short')
    print()
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r',
         os.path.join(REPO_DIR, 'requirements-aga.txt')],
        check=True,
    )
    print('Dependencies installed.')
else:
    REPO_DIR = os.getcwd()
    while REPO_DIR != '/' and not os.path.isdir(os.path.join(REPO_DIR, '.git')):
        REPO_DIR = os.path.dirname(REPO_DIR)
    print('Local kernel; assuming requirements-aga.txt is installed.')
    print('Repository root:', REPO_DIR)

## 2. Imports

Cached `oulad` modules are dropped first, because a `git reset` in step 1 cannot change code
that Python has already imported.

In [ ]:
import os
import sys
from datetime import timedelta

REPO_DIR = '/content/oulad' if os.path.isdir('/content/oulad') else REPO_DIR
SRC_DIR = os.path.join(REPO_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Python caches imported modules, so a git reset in step 1 does not reach code
# that was already imported earlier in this session. Drop it and re-import.
for name in [m for m in sys.modules if m == 'oulad' or m.startswith('oulad.')]:
    del sys.modules[name]

import matplotlib.pyplot as plt
import pandas as pd
from neo4j import GraphDatabase
from graphdatascience.session import (
    AlgorithmCategory,
    AuraAPICredentials,
    DbmsConnectionInfo,
    GdsSessions,
)

import graphdatascience
from oulad.credentials import (
    AGA_SECRETS,
    ETL_SECRETS,
    MissingCredentialsError,
    aura_instance_id,
    load_credentials,
)
from oulad.logger import get_logger

print('graphdatascience', graphdatascience.__version__)
print('pandas          ', pd.__version__)
print('repository      ', REPO_DIR)

## 3. Credentials and the database connection

Both groups are required here: `ETL_SECRETS` to reach the database, `AGA_SECRETS` to open
the session. Asking for both at once means one clear error listing everything missing,
rather than failing again three cells later.

In [ ]:
logger = get_logger(REPO_DIR)

try:
    print('resolved from:', load_credentials(logger, required=ETL_SECRETS + AGA_SECRETS))
except MissingCredentialsError as error:
    raise SystemExit(f'\n{error}\n\nAdd the missing secrets in the sidebar, switch on '
                     'Notebook access, then re-run this cell.')

NEO4J_URI = os.environ['NEO4J_URI']
NEO4J_USERNAME = os.environ['NEO4J_USERNAME']
NEO4J_PASSWORD = os.environ['NEO4J_PASSWORD']
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE') or None
AURA_INSTANCE_ID = aura_instance_id(logger)

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()
print('connected to AuraDB, instance', AURA_INSTANCE_ID)

sessions = GdsSessions(api_credentials=AuraAPICredentials(
    os.environ['AURA_CLIENT_ID'],
    os.environ['AURA_CLIENT_SECRET'],
    os.environ['AURA_PROJECT_ID'],
))
existing = sessions.list()
print(f'{len(existing)} session(s) already in this project:',
      [s.name for s in existing] or 'none')

## 4. Choose the scope

The whole graph is 8.46M `REVIEWED_MATERIAL` relationships. Similarity over all of it is a
big job, so this defaults to a **single module**, which is plenty to see the effect. Widen
`MODULES` when you want the full picture.

| Module | Students | Materials | Interactions |
| --- | --- | --- | --- |
| `FFF` | 6,799 | 1,950 | 3,248,703 |
| `DDD` | 5,407 | 1,672 | 1,866,156 |
| `BBB` | 6,484 | 1,152 | 1,139,085 |
| `CCC` | 3,852 | 400 | 884,889 |
| `EEE` | 2,634 | 321 | 758,220 |
| `GGG` | 2,359 | 367 | 281,277 |
| `AAA` | 702 | 406 | 280,990 |

Every material belongs to exactly one module, so these partition the graph cleanly.

In [ ]:
MODULES = ['BBB']          # e.g. ['BBB', 'CCC'] or the full list to widen the scope

SCOPE_QUERY = '''
MATCH (s:Student)-[r:REVIEWED_MATERIAL]->(m:EducationalMaterial)<-[:HAS_MATERIAL]-(c:Course)
WHERE c.codeModule IN $modules
RETURN count(DISTINCT s) AS students,
       count(DISTINCT m) AS materials,
       count(r) AS interactions
'''

with driver.session(database=NEO4J_DATABASE) as session:
    scope = session.run(SCOPE_QUERY, modules=MODULES).single()

student_count, material_count = scope['students'], scope['materials']
node_count = student_count + material_count
relationship_count = scope['interactions']

print(f'modules {MODULES}')
print(f'  {student_count:,} students + {material_count:,} materials '
      f'= {node_count:,} nodes')
print(f'  {relationship_count:,} REVIEWED_MATERIAL relationships')

## 5. Size and open the session

`estimate` turns the projection's shape into a memory tier, and `get_or_create` is
idempotent on the session name — re-running attaches to the session you already have
instead of starting a second one.

The name carries `AURA_CLIENT_ID` because session names must be unique within an Aura
project, and the client id is what differs between people sharing an instance.

In [ ]:
memory = sessions.estimate(
    node_count=node_count,
    relationship_count=relationship_count,
    algorithm_categories=[
        AlgorithmCategory.SIMILARITY,
        AlgorithmCategory.COMMUNITY_DETECTION,
        AlgorithmCategory.CENTRALITY,
    ],
    node_label_count=2,               # Student, EducationalMaterial
    node_property_count=1,            # id
    relationship_property_count=1,    # sumClick
)
print('estimated memory:', memory)

# Truncated: the full client id is half a credential pair, and notebook
# output gets pasted into issues. Eight characters stays unique per person.
SESSION_NAME = f"oulad-cohorts-{os.environ['AURA_CLIENT_ID'][:8]}"

gds = sessions.get_or_create(
    session_name=SESSION_NAME,
    memory=memory,
    db_connection=DbmsConnectionInfo(
        aura_instance_id=AURA_INSTANCE_ID,
        username=NEO4J_USERNAME,
        password=NEO4J_PASSWORD,
        database=NEO4J_DATABASE,
    ),
    # A backstop: if this notebook is abandoned, the session stops billing on its own.
    ttl=timedelta(hours=2),
)
print('session ready:', SESSION_NAME)

## 6. Project the graph into the session

A *remote* projection: the query runs on AuraDB and streams the result into the session, so
the data never round-trips through this notebook. `gds.graph.project.remote` inside the query
is what marks each row as a relationship to project.

The query asks for `sumClick` as a relationship property, but **on a real run against
`graphdatascience` 2.0a5 it did not arrive** — no error, the property simply was not there.
That matters because GDS then treats a missing weight as zero rather than raising, which is
how a weighted centrality can quietly return all zeros. The cell prints what the projection
actually holds so you can see the difference between what was asked for and what landed;
step 11 sidesteps it by not depending on a projected weight at all.

In [ ]:
GRAPH_NAME = 'oulad-engagement'

PROJECTION_QUERY = '''
MATCH (s:Student)-[r:REVIEWED_MATERIAL]->(m:EducationalMaterial)<-[:HAS_MATERIAL]-(c:Course)
WHERE c.codeModule IN $modules
RETURN gds.graph.project.remote(s, m, {
    sourceNodeLabels: labels(s),
    targetNodeLabels: labels(m),
    sourceNodeProperties: s { .id },
    targetNodeProperties: m { .id },
    relationshipType: type(r),
    relationshipProperties: r { .sumClick }
})
'''

result = gds.graph.project.cypher(
    graph_name=GRAPH_NAME,
    query=PROJECTION_QUERY,
    query_parameters={'modules': MODULES},
    overwrite=True,
)
G = gds.graph.get(GRAPH_NAME)
print(f'projected {G.node_count():,} nodes and {G.relationship_count():,} relationships')

# What actually arrived. A remote projection can drop a property from the
# config map without complaining, and a missing relationship weight then
# scores every node 0 instead of raising -- so check rather than assume.
print('node properties        :', G.node_properties())
print('relationship properties:', G.relationship_properties())

## 7. Link comparable learners

Node similarity compares nodes by the neighbours they share. `REVIEWED_MATERIAL` points
from students to materials, so only students have outgoing relationships — which means every
pair it produces is a **student pair**, scored by the Jaccard overlap of the materials they
touched. No filtering needed.

`top_k=10` keeps the ten strongest partners per student, so the similarity graph stays sparse
enough for Louvain to find structure in.

In [ ]:
similarity = gds.node_similarity.mutate(
    G,
    mutate_relationship_type='SIMILAR_TO',
    mutate_property='similarity',
    relationship_types=['REVIEWED_MATERIAL'],
    similarity_metric='JACCARD',
    top_k=10,
)
print(f'{similarity.relationships_written:,} SIMILAR_TO relationships written')

sample = gds.graph.relationships.stream(
    G, relationship_types=['SIMILAR_TO'], relationship_properties=['similarity'])
print('\nstrongest pairs:')
print(sample.nlargest(5, 'similarity').to_string(index=False))

## 8. Fall out the cohorts

Louvain over the `SIMILAR_TO` graph only. The bipartite `REVIEWED_MATERIAL` edges are
excluded deliberately: including them would cluster students *with* materials, and the
question here is which students resemble each other.

`mutate` rather than `stream`, so the cohort becomes a node property we can write back in
step 12.

In [ ]:
louvain = gds.louvain.mutate(
    G,
    mutate_property='cohort',
    relationship_types=['SIMILAR_TO'],
    node_labels=['Student'],
)
print(f'{louvain.community_count:,} cohorts, modularity {louvain.modularity:.4f}')

def as_frame(streamed, name):
    """Normalise a single-property stream to columns nodeId and `name`.

    The value column's name varies across client versions, so it is taken
    positionally rather than assumed.
    """
    value_column = [c for c in streamed.columns if c != 'nodeId'][-1]
    return streamed.rename(columns={value_column: name})[['nodeId', name]]

cohorts = as_frame(
    gds.graph.node_properties.stream(G, 'cohort', node_labels=['Student']), 'cohort')
student_ids = as_frame(
    gds.graph.node_properties.stream(G, 'id', node_labels=['Student']), 'studentId')
students = cohorts.merge(student_ids, on='nodeId')
students['studentId'] = students['studentId'].astype(int)

sizes = students['cohort'].value_counts()
print(f'\nlargest cohorts (of {len(sizes)}):')
print(sizes.head(10).to_string())
print(f'\ncohorts with a single student: {(sizes == 1).sum():,}')

## 9. Do the cohorts track outcomes?

Outcomes live on the registration, not the student: the path is
`(:Student)-[:WAS_REGISTERED]->(:StudentRegistration)-[:CONTAINS_COURSE]->(:Course)`, and
`finalResult` is a property of that last relationship.

A student can register for the same module in more than one presentation, so this keeps one
row per student. If the cohorts carry any signal, the Pass/Distinction share will differ
between them by more than a couple of points.

In [ ]:
OUTCOME_QUERY = '''
MATCH (s:Student)-[:WAS_REGISTERED]->(:StudentRegistration)-[r:CONTAINS_COURSE]->(c:Course)
WHERE c.codeModule IN $modules
RETURN s.id AS studentId, r.finalResult AS finalResult
'''

with driver.session(database=NEO4J_DATABASE) as session:
    outcomes = pd.DataFrame(
        session.run(OUTCOME_QUERY, modules=MODULES).data()
    ).drop_duplicates(subset='studentId')

TOP_N = 8
biggest = students['cohort'].value_counts().head(TOP_N).index
merged = students[students['cohort'].isin(biggest)].merge(outcomes, on='studentId')

share = (pd.crosstab(merged['cohort'], merged['finalResult'], normalize='index') * 100)
order = [c for c in ['Distinction', 'Pass', 'Fail', 'Withdrawn'] if c in share.columns]
share = share[order].round(1)
counts = merged['cohort'].value_counts()
share.insert(0, 'students', counts[share.index])

print(f'outcome mix by cohort, {TOP_N} biggest (% of each cohort)')
print(share.to_string())

overall = (outcomes['finalResult'].value_counts(normalize=True) * 100).round(1)
print('\nwhole module for comparison:')
print(overall[order].to_string())

dropped = outcomes['studentId'].nunique() - students['studentId'].nunique()
print(f'\nregistered students with no cohort (never touched a material in scope): '
      f'{dropped:,}')

ax = share[order].plot(kind='barh', stacked=True, figsize=(9, 0.5 * len(share) + 2))
ax.set_xlabel('% of cohort')
ax.set_ylabel('cohort')
ax.set_title(f'Outcome mix by engagement cohort — module(s) {", ".join(MODULES)}')
ax.legend(title='finalResult', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 10. Who the cohorts leave out

A student with no `REVIEWED_MATERIAL` relationship has nothing to project, so they cannot be
similar to anyone and cannot land in a cohort. That is not a rounding error: this group is
where the withdrawals concentrate, which means **every cohort result above is conditional on
having engaged at all.**

Read the two tables together. Any claim that cohorts predict outcomes is a claim about
students who showed up, and the students who did not are the ones a real intervention would
care about most.

In [ ]:
ENGAGEMENT_QUERY = '''
MATCH (s:Student)-[:WAS_REGISTERED]->(:StudentRegistration)-[r:CONTAINS_COURSE]->(c:Course)
WHERE c.codeModule IN $modules
WITH DISTINCT s, r.finalResult AS finalResult
OPTIONAL MATCH (s)-[i:REVIEWED_MATERIAL]->(:EducationalMaterial)<-[:HAS_MATERIAL]-(c2:Course)
WHERE c2.codeModule IN $modules
WITH s, finalResult, count(i) AS interactions
RETURN CASE WHEN interactions = 0 THEN 'never engaged' ELSE 'engaged' END AS engagement,
       finalResult,
       count(*) AS students
'''

with driver.session(database=NEO4J_DATABASE) as session:
    engagement = pd.DataFrame(
        session.run(ENGAGEMENT_QUERY, modules=MODULES).data())

split = engagement.pivot_table(
    index='engagement', columns='finalResult', values='students',
    aggfunc='sum', fill_value=0)
split = split[[c for c in order if c in split.columns]]
split['total'] = split.sum(axis=1)
print('outcome counts by whether the student engaged at all')
print(split.to_string())

pct = (split.drop(columns='total')
       .div(split['total'], axis=0) * 100).round(1)
print('\nas a percentage of each group')
print(pct.to_string())

## 11. Which materials pull the attention?

Two different measures, deliberately kept apart:

- **interactions** — degree centrality with `orientation='REVERSE'`, so it counts incoming
  `REVIEWED_MATERIAL` relationships and therefore measures materials rather than students.
  This is the graph-native number.
- **clicks** — `sum(sumClick)` straight from AuraDB.

Visits and intensity are not the same thing, and the split between them is the interesting
part: a page everyone opens once looks nothing like one a few people hammer.

The degree call is unweighted on purpose. Weighting it by `sumClick` scored every material
zero on a real run, because the remote projection had not carried the relationship property —
and GDS treats a missing weight as zero rather than raising. Step 6 now prints what the
projection actually holds, so that failure is visible where it happens.

In [ ]:
# Unweighted on purpose. Passing relationship_weight_property='sumClick' scored
# every material 0 on a real run, because the remote projection had not carried
# the property -- and a missing weight is not an error, it is a silent zero.
# Counting interactions needs no property and cannot fail that way.
attention = as_frame(gds.degree_centrality.stream(
    G,
    orientation='REVERSE',                  # incoming, so this measures materials
    relationship_types=['REVIEWED_MATERIAL'],
    node_labels=['EducationalMaterial'],
), 'interactions')
attention = attention.merge(
    as_frame(gds.graph.node_properties.stream(
        G, 'id', node_labels=['EducationalMaterial']), 'materialId'),
    on='nodeId',
)
attention['materialId'] = attention['materialId'].astype(int)

# Click totals come from AuraDB, where sumClick certainly exists.
CLICKS_QUERY = '''
MATCH (s:Student)-[r:REVIEWED_MATERIAL]->(m:EducationalMaterial)<-[:HAS_MATERIAL]-(c:Course)
WHERE c.codeModule IN $modules
RETURN m.id AS materialId, m.activityType AS activityType,
       count(DISTINCT s) AS students, sum(r.sumClick) AS clicks
'''
with driver.session(database=NEO4J_DATABASE) as session:
    clicks = pd.DataFrame(session.run(CLICKS_QUERY, modules=MODULES).data())

materials = attention.merge(clicks, on='materialId')
print('most-visited materials')
print(materials.nlargest(15, 'interactions')[
    ['materialId', 'activityType', 'students', 'interactions', 'clicks']
].to_string(index=False))

by_type = (materials.groupby('activityType')[['interactions', 'clicks']].sum()
           .assign(materials=materials.groupby('activityType').size())
           .sort_values('clicks', ascending=False))
by_type['clicksPerMaterial'] = (by_type['clicks'] / by_type['materials']).round(0).astype(int)
print('\nattention by activity type')
print(by_type.to_string())

## 12. Write the cohorts back to AuraDB

`node_properties.write` pushes a session property onto the matching nodes in the database.
The dict form renames on the way out, so the graph gets `engagementCohort` rather than the
terser name used inside the session.

After this, cohorts are queryable in AuraDB without a session:

```cypher
MATCH (s:Student) WHERE s.engagementCohort IS NOT NULL
RETURN s.engagementCohort AS cohort, count(*) AS students
ORDER BY students DESC LIMIT 10
```

In [ ]:
written = gds.graph.node_properties.write(
    G,
    {'cohort': 'engagementCohort'},
    node_labels=['Student'],
)
print(f'{written.properties_written:,} students labelled with a cohort')

with driver.session(database=NEO4J_DATABASE) as session:
    check = session.run('''
        MATCH (s:Student) WHERE s.engagementCohort IS NOT NULL
        RETURN count(*) AS labelled, count(DISTINCT s.engagementCohort) AS cohorts
    ''').single()
print(f"verified in AuraDB: {check['labelled']:,} students across "
      f"{check['cohorts']:,} cohorts")

## 13. Clean up

**Run this even if something above failed.** The session is billed compute and independent
of your AuraDB instance; the 2-hour TTL from step 5 is a backstop, not a substitute.

`sessions.list()` at the end should no longer show this session.

In [ ]:
try:
    G.drop()
    print('projection dropped')
except Exception as error:
    print('projection already gone:', error)

gds.delete()
driver.close()
print('session deleted, driver closed')
print('sessions still in the project:', [s.name for s in sessions.list()] or 'none')

## Next

`aga_fastpath_journeys.ipynb` takes the same data as a *sequence* rather than a set: each
student's VLE clicks in date order, embedded with FastPath, so two learners can be compared
on the shape of their journey rather than on which materials they happened to touch. That
needs an event-chain model in the graph, which the notebook builds first.